# 01 - Exploracion de un PDF de PTR

Notebook de apoyo para disenar el parser de la **Fase 3**. Descarga un PDF de PTR e inspecciona su texto y tablas para entender el formato antes de escribir el extractor de transacciones.

Requiere: pip install -r ../requirements.txt (pdfplumber).

In [ ]:
import sys, urllib.request
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
from backend import config

# Ejemplo: Nancy Pelosi, 2026, DocID 20033725
year, doc_id = 2026, '20033725'
url = config.PTR_PDF_URL.format(year=year, doc_id=doc_id)
out = config.RAW_DIR / 'ptr' / str(year) / (doc_id + '.pdf')
out.parent.mkdir(parents=True, exist_ok=True)
req = urllib.request.Request(url, headers={'User-Agent': config.USER_AGENT})
out.write_bytes(urllib.request.urlopen(req, timeout=60).read())
print('Descargado:', out, out.stat().st_size, 'bytes')

In [ ]:
import pdfplumber

with pdfplumber.open(out) as pdf:
    print('Paginas:', len(pdf.pages))
    print(pdf.pages[0].extract_text()[:1500])

In [ ]:
# Inspeccionar tablas (las transacciones suelen venir tabuladas)
with pdfplumber.open(out) as pdf:
    for i, page in enumerate(pdf.pages):
        for t in page.extract_tables():
            print('--- pagina', i, '---')
            for row in t[:8]:
                print(row)